# 02 — Feature Engineering & Construction de la Target

## Objectifs de ce notebook
1. Définir la **target binaire** (réactivation) avec un split temporel propre
2. Construire **3 familles de features** : RFM modifié, expérience 1ère commande, géo
3. Sortir une table d'apprentissage `customer_features.csv` prête pour la modélisation
4. Préparer 2-3 agrégations pour le dashboard Power BI

---

## 🎯 Décision stratégique : Approche A — Prédiction de réactivation

### Le contexte qui force ce choix
L'exploration de l'étape 1 a révélé un fait critique :
- **97% des clients Olist n'ont qu'une seule commande** (90 557 sur 93 358)
- Seulement 2 573 clients ont 2 commandes
- À peine 228 clients ont 3 commandes ou plus

Ce constat **invalide l'approche RFM classique** habituellement utilisée 
sur ce dataset, dans laquelle la composante "Frequency" serait à 1 pour 
quasiment tous les clients (donc inutilisable comme variable discriminante).

### Définition opérationnelle de la target

> Un client est considéré comme **churné (target = 1)** s'il n'a passé 
> aucune nouvelle commande dans les 180 jours suivant la date de coupure T0.

- **T0 = 2018-02-01** (date de coupure)
- **Fenêtre d'observation** : avant T0 → on construit les caracteristique (feature)
- **Fenêtre de prédiction** : T0 → T0+180j → on observe si le client revient

Cette structure de **split temporel** est essentielle : elle évite tout 
**data leakage** (utiliser des informations futures pour prédire le passé), 
faute classique dans les projets ML débutants.

### Structure temporelle du dataset

```
2016-09-01 ── 2017-01-01 ──────────── 2018-02-01 ──────── 2018-08-01
   │              │                        │                    │
   │  Démarrage   │   Fenêtre observation  │  Fenêtre prédiction│
   │   (exclu)    │     (features 13 mois) │     (180 jours)    │
   │              │                        │                    │
   └─ bruit       └─ comportement client   └─ on regarde si le  │
      volumes        capturé ici              client revient    │
      faibles                                                   │
```

## 1. Setup et reconnexion à la base

On reprend la connexion DuckDB persistante créée à l'étape 1. Tous les CSV 
sont déjà chargés en tables, on n'a donc rien à recharger.

On définit aussi les **constantes temporelles** du projet en haut du 
notebook — c'est une bonne pratique : si demain on veut tester une 
fenêtre de 90j ou 120j, on change juste une seule variable.

In [1]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path

# Configuration affichage
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

# Reconnexion à la base
con = duckdb.connect("../data/olist.duckdb")

# === CONSTANTES DU PROJET ===
T0 = "2018-02-01"                  # Date de coupure
PREDICTION_WINDOW_DAYS = 180       # Fenêtre de prédiction (jours)
OBSERVATION_START = "2017-01-01"   # Début fenêtre d'observation

# Chemin de sortie
DATA_PROCESSED = Path("../data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"✅ Date de coupure T0 : {T0}")
print(f"✅ Fenêtre de prédiction : {PREDICTION_WINDOW_DAYS} jours")
print(f"✅ Début observation : {OBSERVATION_START}")

✅ Date de coupure T0 : 2018-02-01
✅ Fenêtre de prédiction : 180 jours
✅ Début observation : 2017-01-01


## 2. Construction de la cohorte éligible

**Règle d'éligibilité** : un client doit avoir passé **au moins une commande 
livrée entre le 2017-01-01 et le 2018-02-01** pour être inclus dans l'analyse.

### Pourquoi cette règle
- Si un client n'a pas commandé **avant T0**, on n'a aucune feature pour lui 
  → impossible de le scorer.
- On exclut la phase de démarrage (sept-déc 2016) car les volumes sont 
  trop faibles et le comportement client n'est pas représentatif.

### Conséquence sur la volumétrie
On va passer de ~93 358 clients uniques à un sous-ensemble plus restreint 
(probablement ~70 000-80 000 clients). C'est normal et attendu — on troque 
de la volumétrie contre de la **propreté méthodologique**.